# 01 — MVTec AD transistor EDA

Read-only exploratory analysis of the **transistor** category under `data/raw/transistor/`.

This notebook inventories splits, image properties, and mask alignment, then visualizes samples
for a future anomaly-detection baseline. It does **not** train models, define PyTorch `Dataset`
classes, or write anything under `data/raw/`.

## 0. Setup

Resolve the repository root from either the repo directory or `notebooks/`, then fail if the
transistor category is missing.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
CATEGORY = "transistor"
RNG = np.random.default_rng(42)

cwd = Path.cwd().resolve()
if (cwd / "data" / "raw" / CATEGORY).is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / "data" / "raw" / CATEGORY).is_dir():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/raw/transistor. "
        "Start Jupyter from the InspectAI repo root or from notebooks/."
    )

RAW_ROOT = REPO_ROOT / "data" / "raw"
CATEGORY_DIR = RAW_ROOT / CATEGORY
FIGURE_DIR = REPO_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

assert CATEGORY_DIR.is_dir(), f"Missing category directory: {CATEGORY_DIR}"
print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"CATEGORY_DIR  = {CATEGORY_DIR}")

## 1. Validate category path

Confirm the MVTec split folders exist before counting files.

In [ ]:
children = sorted(p.name for p in CATEGORY_DIR.iterdir() if p.is_dir())
other_files = sorted(p.name for p in CATEGORY_DIR.iterdir() if p.is_file())
print("Immediate directories:", children)
print("Non-image metadata files:", other_files)

expected = {"train", "test", "ground_truth"}
missing = expected - set(children)
extra = set(children) - expected
assert not missing, f"Missing split folders: {sorted(missing)}"
print("Expected splits present:", sorted(expected))
if extra:
    print("Unexpected extra directories:", sorted(extra))

## 2. Dynamic structure discovery

Class / defect names come from directory listing, not a hardcoded list, so the same pattern
can later apply to other MVTec categories.

In [ ]:
def list_class_dirs(split_dir: Path) -> list[Path]:
    if not split_dir.is_dir():
        return []
    return sorted(p for p in split_dir.iterdir() if p.is_dir())


def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTS


def list_images(class_dir: Path) -> list[Path]:
    return sorted(p for p in class_dir.iterdir() if is_image_file(p))


train_dir = CATEGORY_DIR / "train"
test_dir = CATEGORY_DIR / "test"
gt_dir = CATEGORY_DIR / "ground_truth"

train_classes = [p.name for p in list_class_dirs(train_dir)]
test_classes = [p.name for p in list_class_dirs(test_dir)]
gt_classes = [p.name for p in list_class_dirs(gt_dir)]

print("train classes:", train_classes)
print("test classes: ", test_classes)
print("gt classes:   ", gt_classes)

defect_types = [name for name in test_classes if name != "good"]
print("discovered defect types:", defect_types)

## 3. Inventory counts

MVTec AD is one-class: training contains only `good` images. Test includes `good` plus each
defect type. Masks exist only for defective test images.

In [ ]:
def count_images(split_name: str) -> pd.Series:
    split_path = CATEGORY_DIR / split_name
    counts = {
        class_dir.name: len(list_images(class_dir))
        for class_dir in list_class_dirs(split_path)
    }
    return pd.Series(counts, dtype="int64").sort_index()


train_counts = count_images("train")
test_counts = count_images("test")
gt_counts = count_images("ground_truth")

n_train_good = int(train_counts.get("good", 0))
n_test_good = int(test_counts.get("good", 0))
n_test_defective = int(test_counts.drop(labels=["good"], errors="ignore").sum())

print("train counts")
print(train_counts.to_string())
print("\ntest counts")
print(test_counts.to_string())
print("\nground_truth counts")
print(gt_counts.to_string())
print()
print(f"normal training images: {n_train_good}")
print(f"normal test images:     {n_test_good}")
print(f"defective test images:  {n_test_defective}")
print("test images per defect type:")
print(test_counts.drop(labels=["good"], errors="ignore").to_string())
print("ground-truth masks per defect type:")
print(gt_counts.to_string())

## 4. Image properties

Record width, height, channels, Pillow mode, and file suffix for every image and mask.
Consistent 1024×1024 RGB photos with grayscale masks is the expected MVTec pattern; we verify
it rather than assume it.

In [ ]:
def image_metadata(path: Path) -> dict:
    with Image.open(path) as img:
        width, height = img.size
        mode = img.mode
        channels = len(img.getbands())
        fmt = (img.format or path.suffix.lstrip(".")).lower()
    return {
        "width": width,
        "height": height,
        "channels": channels,
        "mode": mode,
        "format": fmt,
    }


def collect_files(kind: str) -> list[tuple[str, str, Path]]:
    # kind is 'image' (train/test) or 'mask' (ground_truth).
    rows: list[tuple[str, str, Path]] = []
    splits = ["ground_truth"] if kind == "mask" else ["train", "test"]
    for split in splits:
        for class_dir in list_class_dirs(CATEGORY_DIR / split):
            for path in list_images(class_dir):
                rows.append((split, class_dir.name, path))
    return rows


image_files = collect_files("image")
mask_files = collect_files("mask")

prop_rows = []
for split, label, path in image_files + mask_files:
    meta = image_metadata(path)
    prop_rows.append(
        {
            "split": split,
            "label": label,
            "kind": "mask" if split == "ground_truth" else "image",
            "path": path,
            **meta,
        }
    )
props_df = pd.DataFrame(prop_rows)

print("Unique (kind, width, height, channels, mode, format) combinations:")
print(
    props_df.groupby(["kind", "width", "height", "channels", "mode", "format"])
    .size()
    .rename("count")
    .reset_index()
    .to_string(index=False)
)

image_shapes = props_df.loc[props_df["kind"] == "image", ["width", "height", "channels"]].drop_duplicates()
mask_shapes = props_df.loc[props_df["kind"] == "mask", ["width", "height", "channels"]].drop_duplicates()
print("\nImage dimensions consistent:", len(image_shapes) == 1)
print("Mask dimensions consistent: ", len(mask_shapes) == 1)

## 5. Mask pairing / consistency

Defective test images (`label != good`) must have a matching `stem_mask` file in
`ground_truth/<defect>/`. Normal (`good`) images must not require a mask.

In [ ]:
def mask_stem(path: Path) -> str:
    stem = path.stem
    return stem[: -len("_mask")] if stem.endswith("_mask") else stem


def expected_mask_path(label: str, image_path: Path) -> Path:
    return CATEGORY_DIR / "ground_truth" / label / f"{image_path.stem}_mask{image_path.suffix}"


missing_masks: list[str] = []
unmatched_masks: list[str] = []
good_with_masks: list[str] = []

for class_dir in list_class_dirs(test_dir):
    label = class_dir.name
    images = list_images(class_dir)
    if label == "good":
        gt_good = gt_dir / "good"
        if gt_good.is_dir():
            for mask_path in list_images(gt_good):
                good_with_masks.append(str(mask_path.relative_to(CATEGORY_DIR)))
        continue

    image_stems = {p.stem for p in images}
    gt_class = gt_dir / label
    mask_stems = {mask_stem(p) for p in list_images(gt_class)} if gt_class.is_dir() else set()

    for stem in sorted(image_stems - mask_stems):
        missing_masks.append(f"test/{label}/{stem}")
    for stem in sorted(mask_stems - image_stems):
        unmatched_masks.append(f"ground_truth/{label}/{stem}_mask")

print(f"Missing masks for defective test images: {len(missing_masks)}")
for item in missing_masks:
    print("  ", item)
print(f"Unmatched extra masks: {len(unmatched_masks)}")
for item in unmatched_masks:
    print("  ", item)
print(f"Unexpected masks for good images: {len(good_with_masks)}")
for item in good_with_masks:
    print("  ", item)

pairing_ok = not missing_masks and not unmatched_masks and not good_with_masks
print("Pairing consistent:" , pairing_ok)

## 6. Summary DataFrame

One row per train/test image. `mask_path` is set only for defective test images.

In [ ]:
records = []
for split, label, path in image_files:
    meta = image_metadata(path)
    is_anomaly = split == "test" and label != "good"
    mask_path = expected_mask_path(label, path) if is_anomaly else None
    if mask_path is not None and not mask_path.is_file():
        mask_path = None
    records.append(
        {
            "split": split,
            "label": label,
            "is_anomaly": is_anomaly,
            "path": str(path.relative_to(REPO_ROOT)),
            "mask_path": None
            if mask_path is None
            else str(mask_path.relative_to(REPO_ROOT)),
            **meta,
        }
    )

inventory = pd.DataFrame.from_records(records)
inventory = inventory.sort_values(["split", "label", "path"]).reset_index(drop=True)

print(inventory.head())
print()
print("rows:", len(inventory))
print("value_counts split x label:")
print(inventory.groupby(["split", "label"]).size().rename("n").to_string())
print()
print(inventory[["width", "height", "channels"]].describe().to_string())

## 7. Class-balance plot

Training is heavily `good`-only. Each defect type has a small test set compared with normal
train/test images.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

split_label_counts = (
    inventory.groupby(["split", "label"]).size().unstack(fill_value=0)
)
split_label_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Images per split and class")
axes[0].set_xlabel("split")
axes[0].set_ylabel("count")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="label", bbox_to_anchor=(1.02, 1), loc="upper left")

defect_counts = (
    inventory.loc[inventory["is_anomaly"]].groupby("label").size().sort_index()
)
axes[1].bar(defect_counts.index.astype(str), defect_counts.to_numpy())
axes[1].set_title("Test samples per defect type")
axes[1].set_xlabel("defect type")
axes[1].set_ylabel("count")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
balance_path = FIGURE_DIR / "transistor_class_balance.png"
fig.savefig(balance_path, dpi=120, bbox_inches="tight")
print(f"saved {balance_path.relative_to(REPO_ROOT)}")
plt.show()

## 8. Representative visualization

Fixed RNG seed. Show normal train/test samples, then one defective image per discovered
defect type with its binary mask and overlay. Helpers stay in this notebook (later candidates
for `src/inspectai/visualization/`).

In [ ]:
def load_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(img.convert("RGB"))


def load_mask(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr > 0


def overlay_mask(image: np.ndarray, mask: np.ndarray, color=(255, 0, 0), alpha=0.45) -> np.ndarray:
    out = image.astype(np.float32).copy()
    color_arr = np.array(color, dtype=np.float32)
    out[mask] = (1.0 - alpha) * out[mask] + alpha * color_arr
    return np.clip(out, 0, 255).astype(np.uint8)


def sample_paths(frame: pd.DataFrame, n: int) -> list[Path]:
    idx = RNG.choice(len(frame), size=min(n, len(frame)), replace=False)
    return [REPO_ROOT / frame.iloc[int(i)]["path"] for i in idx]


def show_image_row(paths: list[Path], title: str) -> None:
    fig, axes = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4))
    if len(paths) == 1:
        axes = [axes]
    for ax, path in zip(axes, paths):
        ax.imshow(load_rgb(path))
        ax.set_title(path.name)
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    safe = title.lower().replace(" ", "_").replace("/", "_")
    out = FIGURE_DIR / f"transistor_{safe}.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    print(f"saved {out.relative_to(REPO_ROOT)}")
    plt.show()


train_good = inventory[(inventory["split"] == "train") & (inventory["label"] == "good")]
test_good = inventory[(inventory["split"] == "test") & (inventory["label"] == "good")]
show_image_row(sample_paths(train_good, 3), "Normal training images")
show_image_row(sample_paths(test_good, 3), "Normal test images")

defect_rows = []
for defect in defect_types:
    subset = inventory[(inventory["split"] == "test") & (inventory["label"] == defect)]
    row = subset.iloc[int(RNG.integers(0, len(subset)))]
    defect_rows.append(row)

n = len(defect_rows)
fig, axes = plt.subplots(n, 3, figsize=(10, 3.4 * n))
if n == 1:
    axes = np.array([axes])
for ax_row, row in zip(axes, defect_rows):
    image = load_rgb(REPO_ROOT / row["path"])
    mask = load_mask(REPO_ROOT / row["mask_path"])
    ax_row[0].imshow(image)
    ax_row[0].set_title(f"{row['label']} / {Path(row['path']).name}")
    ax_row[1].imshow(mask, cmap="gray")
    ax_row[1].set_title("binary mask")
    ax_row[2].imshow(overlay_mask(image, mask))
    ax_row[2].set_title("overlay")
    for ax in ax_row:
        ax.axis("off")
fig.suptitle("Defective images, masks, and overlays")
fig.tight_layout()
defect_fig = FIGURE_DIR / "transistor_defect_overlays.png"
fig.savefig(defect_fig, dpi=120, bbox_inches="tight")
print(f"saved {defect_fig.relative_to(REPO_ROOT)}")
plt.show()

## 9. Defect-geometry notes (from masks, no model)

Foreground fraction and bounding-box size (relative to 1024×1024) indicate how much of the
frame a defect occupies. Small lead defects vs large `misplaced` regions have different
resize / patch-size implications.

In [ ]:
geo_rows = []
anomaly = inventory[inventory["is_anomaly"]].copy()
for _, row in anomaly.iterrows():
    mask = load_mask(REPO_ROOT / row["mask_path"])
    h, w = mask.shape
    ys, xs = np.where(mask)
    frac = float(mask.mean())
    if len(xs) == 0:
        bbox_w = bbox_h = cy = cx = 0.0
    else:
        bbox_w = (xs.max() - xs.min() + 1) / w
        bbox_h = (ys.max() - ys.min() + 1) / h
        cy = float(ys.mean() / h)
        cx = float(xs.mean() / w)
    geo_rows.append(
        {
            "label": row["label"],
            "path": row["path"],
            "fg_fraction": frac,
            "bbox_w_frac": bbox_w,
            "bbox_h_frac": bbox_h,
            "centroid_y": cy,
            "centroid_x": cx,
        }
    )

geometry = pd.DataFrame(geo_rows)
summary = geometry.groupby("label")[["fg_fraction", "bbox_w_frac", "bbox_h_frac"]].agg(
    ["min", "median", "max"]
)
print(summary.to_string())
print()
print(
    "Smallest defect (fg_fraction):",
    geometry.loc[geometry["fg_fraction"].idxmin(), ["label", "fg_fraction", "path"]].to_dict(),
)
print(
    "Largest defect (fg_fraction):",
    geometry.loc[geometry["fg_fraction"].idxmax(), ["label", "fg_fraction", "path"]].to_dict(),
)

fig, ax = plt.subplots(figsize=(7, 4))
labels = list(geometry["label"].unique())
data = [geometry.loc[geometry["label"] == lab, "fg_fraction"].to_numpy() for lab in labels]
ax.boxplot(data, tick_labels=labels)
ax.set_ylabel("mask foreground fraction")
ax.set_title("Defect size by type (pixel fraction of 1024x1024)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
geo_fig = FIGURE_DIR / "transistor_defect_size.png"
fig.savefig(geo_fig, dpi=120, bbox_inches="tight")
print(f"saved {geo_fig.relative_to(REPO_ROOT)}")
plt.show()

## 10. Observations (for future anomaly detection)

These notes follow the inventory, pairing check, geometry table, and sample images in this
notebook. They are not experiment results.

- **Class imbalance.** Training is one-class: 213 `good` images and zero defect examples.
  The test set has 60 `good` images and 10 images per discovered defect type
  (`bent_lead`, `cut_lead`, `damaged_case`, `misplaced`). Image-level metrics will be
  dominated by the 60 normals unless they are reported per defect type as well.
- **Image properties.** All RGB photos are 1024×1024 PNG; all masks are 1024×1024
  single-channel PNG. Spatial size is consistent, so a global resize is optional rather
  than required for batching.
- **Mask pairing.** Every defective test image has a matching `*_mask.png`; `good` images
  have no ground-truth folder. Pixel-level evaluation can use a 1:1 image–mask map.
- **Defect size.** `bent_lead` and `cut_lead` occupy on the order of **0.2%–3.6%** of pixels
  (median about 1.0% and 1.5%). `damaged_case` is larger (median about 2.3%, max about 5.4%).
  `misplaced` masks the **entire component** (about 38%–49% of pixels) because the anomaly
  is pose/placement, not a local nick.
- **Defect location.** Lead defects sit on the lower metal pins (often a small white region
  on the mask). `damaged_case` sits on the black plastic body. `misplaced` covers body plus
  leads. A detector that only looks at the package surface would miss lead defects; one that
  only looks at pins would miss case damage.
- **Object alignment.** Good parts are roughly centered on a copper perfboard, but pin
  insertion already bends leads into the holes. `misplaced` samples are rotated or shifted.
  Alignment is structured, not pixel-identical, so a method that assumes a perfectly
  registered template will see residual “anomalies” on every good image.
- **Lighting and background.** The background is a repeating copper board with drilled holes.
  Specular highlights appear on plastic and metal. Background texture is consistent across
  splits, which helps reconstruction / embedding methods, but hole edges can look similar to
  thin lead defects at low resolution.
- **Resize risk.** Downsampling 1024×1024 to 224×224 (about 4.6× per side) can shrink a
  0.2% lead defect to a few pixels. Prefer keeping relatively high resolution, using patches,
  or evaluating at native size for `bent_lead` / `cut_lead`. Aggressive resize is less
  harmful for `misplaced`.
- **Label subtlety.** Some `good` leads are already curved from board insertion. `bent_lead`
  is a more extreme displacement, not “any curve.” That makes this category a better test of
  fine-grained industrial defects than of coarse object presence.

Reusable discovery, inventory, pairing, metadata, and overlay helpers should later move to
`src/inspectai/data/` and `src/inspectai/visualization/`. They stay in this notebook for
Milestone 1.

## 11. Out of scope

This notebook does **not**:

- implement PyTorch `Dataset` classes or training loops
- add Autoencoders, PatchCore, DINO, YOLO, or other detectors
- add APIs or dashboards
- modify any file under `data/raw/`

Figures, if saved, go only to `outputs/figures/`.